In [ ]:
import os
import pandas as pd
import yaml
import pickle

from utils.training_utils import find_specific_variables

import xgboost as xgb

from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

In [ ]:
features = yaml.safe_load(open(os.path.join('..', 'src', 'config', 'feature_config.yaml'), 'r'))

# Modelo para classificação de um produto em promoção

In [ ]:
df = pd.read_csv(os.path.join('..', 'data', 'train_test', 'train_encoded.csv'))

print(df.shape)
df.head()

In [ ]:
seletor = pickle.load(
    open(os.path.join('..', 'models', 'encoders', 'seletor_2.pkl'), 'rb')
)

df_hyperparams = pickle.load(
    open(os.path.join('..', 'models', 'df_metrics_results_tunning.pkl'), 'rb')
)

In [ ]:

feature_target = find_specific_variables(features, 'target', specific_value=True)

In [ ]:
df_treino, df_valid = train_test_split(df, test_size=0.2, random_state=96)

In [ ]:
print(f'Shape Treino: {df_treino.shape}')
print(f'Shape Valid: {df_valid.shape}')

In [ ]:
print(f'% Treino: {df_treino[feature_target[0]].mean()}')
print(f'% Valid: {df_valid[feature_target[0]].mean()}')

In [ ]:
df_hyperparams[df_hyperparams.value == max(df_hyperparams.value)].T

In [ ]:
best_row = df_hyperparams.loc[df_hyperparams['value'].idxmax()]
best_params = best_row.filter(like='params_')
hyper_params = {col.replace('params_', ''): best_params[col] for col in best_params.index}


hyper_params.update({
    'eval_metric': 'auc',
})

In [ ]:
hyper_params

In [ ]:
model = xgb.XGBClassifier(
    **hyper_params,
    random_state=12,
    n_jobs=-1,
    early_stopping_rounds=4
)

model

In [ ]:
model.fit(
    df_treino[seletor.features].values,
    df_treino[feature_target].values,
    eval_set=[(df_valid[seletor.features].values, df_valid[feature_target].values)],
    verbose=True
)

In [ ]:
pickle.dump(
    model, 
    open(os.path.join('..', 'models', 'predictors', 'model.pkl'), 'wb')
)